In [3]:
import pandas as pd

In [ ]:
full_df_collones_trop_vides = []

def get_columns_emptyness(df):
    df_len = df.shape[0]
    columns_emptyness_infos = []
    
    for column in df.columns.to_list():
        # identification du pourcentage de vide pour chaques colonnes
        column_name = column
        empty_sum = df[column].isna().sum()
        empty_pourcentage = empty_sum / df_len * 100

        column_emptytiness_info = {
            "column_name" : column_name,
            "empty_pourcentage" : empty_pourcentage
        }
        columns_emptyness_infos.append(column_emptytiness_info)

    # identifacation des collones trop vides
    seuil = 95
    
    for column in columns_emptyness_infos:
        if column['empty_pourcentage'] >= seuil :
            full_df_collones_trop_vides.append(column)


In [ ]:
chunksize = 1_000_000
for i, chunk_df in enumerate(pd.read_csv("../data.0./full_openfoodfacts_products_dataset.csv.gz", 
                                         sep="\t",
                                         encoding="utf-8",
                                         low_memory=False,
                                         chunksize=chunksize,
                                         on_bad_lines="skip")):
    get_columns_emptyness(chunk_df)
    print(f"{i} : is done !")

print(full_df_collones_trop_vides)

0 : is done !
1 : is done !
2 : is done !
3 : is done !
[{'column_name': 'abbreviated_product_name', 'empty_pourcentage': np.float64(99.9884)}, {'column_name': 'generic_name', 'empty_pourcentage': np.float64(98.9737)}, {'column_name': 'packaging', 'empty_pourcentage': np.float64(97.1738)}, {'column_name': 'packaging_tags', 'empty_pourcentage': np.float64(97.17360000000001)}, {'column_name': 'packaging_en', 'empty_pourcentage': np.float64(97.174)}, {'column_name': 'packaging_text', 'empty_pourcentage': np.float64(99.845)}, {'column_name': 'origins', 'empty_pourcentage': np.float64(98.78190000000001)}, {'column_name': 'origins_tags', 'empty_pourcentage': np.float64(98.7825)}, {'column_name': 'origins_en', 'empty_pourcentage': np.float64(98.7826)}, {'column_name': 'manufacturing_places', 'empty_pourcentage': np.float64(98.69)}, {'column_name': 'manufacturing_places_tags', 'empty_pourcentage': np.float64(98.6905)}, {'column_name': 'emb_codes', 'empty_pourcentage': np.float64(98.9626)}, {'c

In [6]:
clean_full_collones_trop_vides = []

for infos in full_df_collones_trop_vides:
    if infos:
        clean_full_collones_trop_vides.append(infos)

df_clean_full_collones_trop_vides = pd.DataFrame(clean_full_collones_trop_vides)

In [7]:
df_clean_full_collones_trop_vides

,column_name,empty_pourcentage
0,abbreviated_product_name,99.988400
1,generic_name,98.973700
2,packaging,97.173800
3,packaging_tags,97.173600
4,packaging_en,97.174000
...,...,...
493,carnitine_100g,99.996915
494,sulphate_100g,99.992802
495,nitrate_100g,99.993624
496,acidity_100g,99.998252


In [10]:
df_clean_full_collones_trop_vides
df_clean_full_collones_trop_vides.to_csv('data/all_empty_colums.csv')

In [45]:

df_stats = df_clean_full_collones_trop_vides.groupby('column_name', as_index=False).agg(['mean', 'std'])
df_stats


column_name empty_pourcentage          
                                           mean       std
0    abbreviated_product_name         99.307852  1.110265
1                acidity_100g         99.999138  0.000615
2             added-salt_100g         99.996140  0.005715
3           added-sugars_100g         99.218924  0.071442
4                   additives         99.999875  0.000150
..                        ...               ...       ...
131            vitamin-e_100g         99.468466  0.456053
132            vitamin-k_100g         99.680529  0.500068
133           vitamin-pp_100g         99.091265  1.375816
134       water-hardness_100g         99.999671  0.000397
135                 zinc_100g         99.345775  0.825632

[136 rows x 3 columns]

In [51]:
colonnes_a_supprimer = []
def validate_empty(df_):
    empty_pourcentage_mean = df_[('empty_pourcentage', 'mean')]
    empty_pourcentage_std = df_[('empty_pourcentage', 'std')]
    variation_coeficient = empty_pourcentage_std / empty_pourcentage_mean
    column_name = df_[('column_name', '')]
 

    if variation_coeficient < 0.1:
        if empty_pourcentage_mean >= 95:
            colonnes_a_supprimer.append(column_name)

df_stats.apply(validate_empty, axis=1)

print(colonnes_a_supprimer)

['abbreviated_product_name', 'acidity_100g', 'added-salt_100g', 'added-sugars_100g', 'additives', 'alcohol_100g', 'allergens_en', 'alpha-linolenic-acid_100g', 'arachidic-acid_100g', 'arachidonic-acid_100g', 'behenic-acid_100g', 'beta-carotene_100g', 'beta-glucan_100g', 'bicarbonate_100g', 'biotin_100g', 'brand_owner', 'butyric-acid_100g', 'caffeine_100g', 'calcium_100g', 'capric-acid_100g', 'caproic-acid_100g', 'caprylic-acid_100g', 'carbohydrates-total_100g', 'carbon-footprint-from-meat-or-fish_100g', 'carbon-footprint_100g', 'carnitine_100g', 'casein_100g', 'cerotic-acid_100g', 'chloride_100g', 'chlorophyl_100g', 'cholesterol_100g', 'choline_100g', 'chromium_100g', 'cities', 'cities_tags', 'cocoa_100g', 'collagen-meat-protein-ratio_100g', 'copper_100g', 'data_quality_errors_tags', 'dihomo-gamma-linolenic-acid_100g', 'docosahexaenoic-acid_100g', 'eicosapentaenoic-acid_100g', 'elaidic-acid_100g', 'emb_codes', 'emb_codes_tags', 'energy-from-fat_100g', 'erucic-acid_100g', 'erythritol_100

In [56]:
colonnes_a_supprimer
with open('data/really_empty_colums.txt', 'w') as file :
    file.write(str(colonnes_a_supprimer))

In [59]:
def remove_empty_cols(df_):
    df_= df_.drop(columns=colonnes_a_supprimer)
    df_.to_csv('no_empty_col_dataset.csv',
                mode="w" if first_chunk else 'a', 
                header=first_chunk, 
                index=False )

chunksize = 1_000_000
first_chunk = True


for i, chunk_df in enumerate(pd.read_csv("../data/full_openfoodfacts_products_dataset.csv.gz", 
                                         sep="\t",
                                         encoding="utf-8",
                                         low_memory=False,
                                         chunksize=chunksize,
                                         on_bad_lines="skip")):
    remove_empty_cols(chunk_df)
    first_chunk = False
    print(f"{i} : is done !")





0 : is done !
1 : is done !
2 : is done !
3 : is done !


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq



chunksize = 1_000_000
writer = None

for i, chunk_df in enumerate(pd.read_csv("no_empty_col_dataset.csv", 
                                         sep="\t",
                                         encoding="utf-8",
                                         low_memory=False,
                                         chunksize=chunksize,
                                         on_bad_lines="skip")):
    
    
    table = pa.Table.from_pandas(chunk_df)

    # Initialiser le writer au premier chunk avec le bon schéma
    if writer is None:
        writer = pq.ParquetWriter("no_empty_col_dataset.parquet", table.schema, compression='snappy')

    # Écrire le chunk
    writer.write_table(table)

    print(f"Chunk {i} écrit dans le Parquet")

# Fermer le writer à la fin
if writer:
    writer.close()





Chunk 0 écrit dans le Parquet
Chunk 1 écrit dans le Parquet
Chunk 2 écrit dans le Parquet
Chunk 3 écrit dans le Parquet
